<a href="https://colab.research.google.com/github/amiralito/Protenix_Colab/blob/main/protenix_v2_single.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protenix-v2 on Colab — single prediction

End-to-end [Protenix-v2](https://github.com/bytedance/Protenix) (ByteDance, AF3-class, ~464M params) inference for **one structure at a time**, using the **ColabFold MSA server** (`api.colabfold.com`) so **no local sequence databases** are needed. For running many complexes in one pass, use the companion **batch** notebook.

**Setup:** Runtime → Change runtime type → **GPU**. Protenix is memory-light for small inputs (≈6 GB at 500 tokens, ≈18 GB at 1000), so a T4 handles monomers/small dimers; use an **A100 (40 GB)** for large complexes / resistosomes (≈67 GB at 2000 tokens).

**Flow:**
1. Run **Setup** cells 1–3 once per session (install + helpers).
2. Run **one** Input cell (4a–4d) matching your case — each shows a form.
3. Set options in **Run settings** (cell 5).
4. Run **MSA + inference** (cell 6). Enable **5b** first if you want dense PAE captured. First run also auto-downloads the weights.
5. Rank (7), **MolView** viewer + scores (8), pLDDT/contacts/PAE (9–9c), **PAE export + ipSAE** (9d–9e), zip & download (10–10b).

Protenix-v2 is efficient: ~5 seeds already beats v1 at 1000 seeds.

## 1. GPU check

In [ ]:
#@title 1. GPU check { display-mode: "form" }
import subprocess
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if not out:
    raise SystemExit('STOP: No GPU. Runtime -> Change runtime type -> GPU.')
print(out)
name, mem = [x.strip() for x in out.split(',')]
mem_gb = int(mem.split()[0]) / 1024
print(f'Detected: {name}  (~{mem_gb:.0f} GB)')
if mem_gb < 15:
    print('WARNING: <15 GB VRAM. Only very small monomers will fit. Prefer A100.')
elif mem_gb < 24:
    print('OK for monomers / small complexes (<~1000 tokens). Large assemblies may OOM -> use A100.')
else:
    print('Plenty of VRAM for large complexes.')

## 2. Install Protenix + tools (run once per session)

In [ ]:
#@title 2. Install Protenix + tools (run once per session) { display-mode: "form" }
import os, sys, subprocess, importlib.util

def sh(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
def have(m):
    return importlib.util.find_spec(m) is not None

# 1) Protenix + light viewers (only if missing, so re-running never disturbs a torch fix).
if not have('protenix'):
    sh('pip -q install --upgrade protenix --index-url https://pypi.org/simple')
else:
    print('protenix already installed - skipping its pip.')
for _p in ('py3Dmol', 'gemmi'):
    if not have(_p):
        sh(f'pip -q install {_p}')
sh('apt-get -qq install -y kalign hmmer >/dev/null 2>&1 || true')

# 2) Online ColabFold MSA server (no local databases).
os.environ['MMSEQS_SERVICE_HOST_URL'] = 'https://api.colabfold.com'

# 3) GPU compute capability via nvidia-smi - NO torch import here, on purpose.
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().splitlines()[0].strip()
os.environ['TORCH_CUDA_ARCH_LIST'] = cc          # '12.0' Blackwell, '8.0' A100, etc.

# 4) Blackwell (sm_120): stock/cu126 wheels lack its kernels, AND the pinned 2.7.1+cu128 we
#    used before covered core ops but was missing some sm_120 kernels (e.g. FlashAttention).
#    So we do NOT auto-install a torch here. On sm_120, install the nightly cu128 build via
#    cell 2c ONCE, restart the runtime, then re-run this cell. (No torch touched here = no overlap.)
if cc.replace('.', '') == '120':
    # Protenix's fused layernorm (fastfold_layer_norm_cuda) is compiled without sm_120 and is
    # gated by LAYERNORM_TYPE, NOT by --enable_fusion. Force native PyTorch layernorm on Blackwell.
    os.environ['LAYERNORM_TYPE'] = 'torch'
    print('** Blackwell (sm_120) detected. **')
    print('   - LAYERNORM_TYPE=torch set (native layernorm; the fused kernel has no sm_120 image).')
    print('   - This cell installs no torch. If you have not already: run cell 2c (nightly cu128),')
    print('     Runtime -> Restart session, then re-run this cell and continue.')

print(f'Setup done | compute_cap {cc} | TORCH_CUDA_ARCH_LIST={cc}')
print('MMSEQS_SERVICE_HOST_URL =', os.environ['MMSEQS_SERVICE_HOST_URL'])
print('Note: do not import torch in this cell. If sm_120 kernels still error, see cell 2c (nightly).')

## 2b. Model weights (only needed if cell 6 fails at "Downloading model checkpoint")

ByteDance serves Protenix weights from a Beijing object store that can return **HTTP 403** for
some non-CN IPs, notably for `protenix-v2`. If `pred` dies at the checkpoint download, run this to
fetch the file directly to where `pred` looks (`/root/checkpoint/`). If it stays blocked, switch
`model_name` in cell 5 to `protenix_base_default_v1.0.0` (Protenix-v1, AF3-class, reachable).

In [ ]:
#@title 2b. Model weights (only needed if cell 6 fails at 'Downloading model checkpoint') { display-mode: "form" }
import os, subprocess

os.makedirs('/root/checkpoint', exist_ok=True)
CKPT_URL = 'https://protenix.tos-cn-beijing.volces.com/checkpoint/protenix-v2.pt'
CKPT_DST = '/root/checkpoint/protenix-v2.pt'

r = subprocess.run(
    ['bash', '-lc',
     f'curl -fL --retry 4 --retry-delay 3 -A "Mozilla/5.0" -o "{CKPT_DST}" "{CKPT_URL}"'],
    capture_output=True, text=True)
sz = os.path.getsize(CKPT_DST) / 1e9 if os.path.exists(CKPT_DST) else 0
if r.returncode == 0 and sz > 0.1:
    print(f'OK: protenix-v2.pt fetched ({sz:.2f} GB). Re-run cell 6 - it will skip the download.')
else:
    print(f'Still blocked (curl rc={r.returncode}). stderr tail:')
    print(r.stderr[-600:])
    print()
    print('Fallback: in cell 5 set model_name = "protenix_base_default_v1.0.0" and re-run cell 6.')


## 2c. Blackwell (sm_120) torch - nightly cu128 build

On a Blackwell card (RTX PRO 6000 / RTX 50-series, sm_120), cell 2 installs **no** torch - run
this once to get the nightly cu128 build, which carries the full sm_120 kernel set (including
FlashAttention, which the pinned stable build was missing). **After it finishes, restart the
runtime** (Runtime -> Restart session, which keeps installed packages + Drive), then re-run from
cell 2. On non-Blackwell cards (T4/L4/A100) you do NOT need this - stock torch already works.

In [ ]:
#@title 2c. Blackwell (sm_120) torch - nightly cu128 build { display-mode: "form" }
import os, subprocess
os.environ['TORCH_CUDA_ARCH_LIST'] = '12.0'
# Nightly cu128 (with deps) - last resort if the pinned stable build ever lacks sm_120 coverage.
subprocess.run('pip install --pre --force-reinstall torch '
               '--index-url https://download.pytorch.org/whl/nightly/cu128',
               shell=True, check=True)
print('\nDONE. Now: Runtime -> Restart session, then re-run from cell 2.')
print('Verify after restart:  import torch; print(torch.cuda.get_arch_list())  -> must include sm_120')


## 3. Helpers (run once)

Defines the working dirs and a small set of functions that assemble the Protenix input JSON
([schema](https://github.com/bytedance/Protenix/blob/main/docs/infer_json_format.md)). The input
cells below just call these and stash the result in the global `JOB_SPEC`.

In [ ]:
#@title 3. Helpers (run once) { display-mode: "form" }
import os, json, glob, time

custom_output_dir = ""  #@param {type:"string"}

WORK      = '/content/protenix_run'
INPUT_DIR = os.path.join(WORK, 'inputs')
MSA_DIR   = os.path.join(WORK, 'msa')
OUT_DIR   = custom_output_dir.strip() or os.path.join(WORK, 'outputs')
for d in (INPUT_DIR, MSA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print('Output dir:', OUT_DIR)

JOB_SPEC = None  # set by an input cell

def _clean_seq(s):
    return ''.join(s.split()).upper()

def protein(seq, count=1, ids=None):
    d = {'sequence': _clean_seq(seq), 'count': int(count)}
    if ids: d['id'] = ids
    return {'proteinChain': d}

def ligand(spec, count=1):
    # spec: 'CCD_ATP'  OR  a SMILES string  OR  'FILE_/path/to/lig.sdf'
    return {'ligand': {'ligand': spec, 'count': int(count)}}

def ion(code, count=1):
    # code is a bare CCD code WITHOUT the CCD_ prefix, e.g. 'MG', 'ZN', 'NA'
    return {'ion': {'ion': code, 'count': int(count)}}

def dna(seq, count=1):
    return {'dnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def rna(seq, count=1):
    return {'rnaSequence': {'sequence': _clean_seq(seq), 'count': int(count)}}

def make_spec(name, entities):
    name = name.strip().replace(' ', '_') or 'job'
    return {'name': name, 'sequences': entities}

def write_json(spec, path=None):
    path = path or os.path.join(INPUT_DIR, spec['name'] + '.json')
    with open(path, 'w') as f:
        json.dump([spec], f, indent=2)   # top level must be a LIST
    print('Wrote', path)
    print(json.dumps([spec], indent=2)[:1200])
    return path

print('Helpers ready. Now run ONE input cell (4a-4d).')

## 3b. (Recommended) Save outputs to Google Drive so a disconnect can't lose them

In [ ]:
#@title 3b. (Recommended) Save outputs to Google Drive so a disconnect can't lose them { display-mode: "form" }
# Colab wipes /content the moment the runtime disconnects. Mount Drive and redirect the
# work dirs so predictions AND the MSA cache persist across sessions.
# Run this AFTER cell 3 and BEFORE cell 6.
from google.colab import drive
import os, shutil, time
_mp = '/content/drive'
if os.path.ismount(_mp):
    print('Drive already mounted - skipping mount.')
else:
    if os.path.isdir(_mp) and os.listdir(_mp):
        # not a live mount but has leftover files that block mounting -> move aside (preserve)
        _bak = f'{_mp}_old_{int(time.time())}'
        shutil.move(_mp, _bak)
        print(f'Moved stale {_mp} -> {_bak} (leftover files, not live Drive).')
    drive.mount(_mp)

WORK      = '/content/drive/MyDrive/protenix_run'
INPUT_DIR = os.path.join(WORK, 'inputs')
MSA_DIR   = os.path.join(WORK, 'msa')
_custom   = custom_output_dir.strip() if 'custom_output_dir' in globals() else ''
OUT_DIR   = _custom or os.path.join(WORK, 'outputs')
for d in (INPUT_DIR, MSA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print('Outputs will persist to:', OUT_DIR)


## 4a. Input — single protein (monomer)

In [ ]:
#@title Monomer { display-mode: "form" }
job_name = "ubiquitin_test"  #@param {type:"string"}
protein_sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"  #@param {type:"string"}

JOB_SPEC = make_spec(job_name, [protein(protein_sequence, 1)])
write_json(JOB_SPEC)

## 4b. Input — multimer (up to 4 distinct chains)

In [ ]:
#@title Multimer { display-mode: "form" }
job_name = "complex"  #@param {type:"string"}
chain_A = ""  #@param {type:"string"}
copies_A = 1  #@param {type:"integer"}
chain_B = ""  #@param {type:"string"}
copies_B = 1  #@param {type:"integer"}
chain_C = ""  #@param {type:"string"}
copies_C = 1  #@param {type:"integer"}
chain_D = ""  #@param {type:"string"}
copies_D = 1  #@param {type:"integer"}

ents = []
for seq, n in [(chain_A, copies_A), (chain_B, copies_B), (chain_C, copies_C), (chain_D, copies_D)]:
    if seq.strip():
        ents.append(protein(seq, n))
assert ents, 'Fill in at least one chain.'
JOB_SPEC = make_spec(job_name, ents)
write_json(JOB_SPEC)

## 4c. Input — homomer (one sequence, N copies)

In [ ]:
#@title Homomer { display-mode: "form" }
job_name = "homomer"  #@param {type:"string"}
protein_sequence = ""  #@param {type:"string"}
num_copies = 6  #@param {type:"slider", min:2, max:12, step:1}

JOB_SPEC = make_spec(job_name, [protein(protein_sequence, num_copies)])
write_json(JOB_SPEC)

## 4d. Input — protein + ligand (+ optional ion)

`ligand_spec` accepts a CCD code prefixed with `CCD_` (e.g. `CCD_ATP`), a **SMILES** string, or a
local file as `FILE_/path/to/lig.sdf` (PDB/SDF/MOL/MOL2, must contain a 3D conformer).
`ion_code` is a bare CCD code with **no** prefix (e.g. `MG`, `ZN`, `NA`). Leave ion blank to skip.

In [ ]:
#@title Protein + ligand { display-mode: "form" }
job_name = "protein_ligand"  #@param {type:"string"}
protein_sequence = ""  #@param {type:"string"}
protein_copies = 1  #@param {type:"integer"}
ligand_spec = "CCD_ATP"  #@param {type:"string"}
ligand_copies = 1  #@param {type:"integer"}
ion_code = ""  #@param {type:"string"}
ion_count = 1  #@param {type:"integer"}

ents = [protein(protein_sequence, protein_copies), ligand(ligand_spec, ligand_copies)]
if ion_code.strip():
    ents.append(ion(ion_code.strip(), ion_count))
JOB_SPEC = make_spec(job_name, ents)
write_json(JOB_SPEC)

## 5. Run settings

`protenix-v2` uses sensible defaults (N_cycle=10, N_step=200). With v2, ~5 seeds is plenty.
Templates and RNA-MSA require **local databases** (pdb_seqres / RNA DBs) and are off here.

In [ ]:
#@title Run settings { display-mode: "form" }
model_name = "protenix_base_default_v1.0.0"  #@param ["protenix-v2", "protenix_base_default_v1.0.0", "protenix_base_20250630_v1.0.0", "protenix_base_default_v0.5.0", "protenix_mini_default_v0.5.0"]
seeds_mode = "list"  #@param ["list", "random_n"]
seeds_list = "101,102,103,104,105"  #@param {type:"string"}
random_n = 5  #@param {type:"slider", min:1, max:25, step:1}
samples_per_seed = 5  #@param {type:"slider", min:1, max:25, step:1}
use_msa = True  #@param {type:"boolean"}
msa_server_mode = "colabfold"  #@param ["colabfold", "protenix"]
use_tfg_guidance = False  #@param {type:"boolean"}
dtype = "bf16"  #@param ["bf16", "fp32"]
enable_speed_mem_opts = True  #@param {type:"boolean"}
use_templates = False  #@param {type:"boolean"}
use_rna_msa = False  #@param {type:"boolean"}

import random
if seeds_mode == "list":
    SEEDS = [int(x) for x in seeds_list.split(',') if x.strip()]
else:
    SEEDS = random.sample(range(1, 100000), int(random_n))
SEEDS_CSV = ','.join(str(s) for s in SEEDS)

CFG = dict(model_name=model_name, seeds_csv=SEEDS_CSV, n_sample=int(samples_per_seed),
           use_msa=use_msa, msa_server_mode=msa_server_mode, use_tfg=use_tfg_guidance,
           dtype=dtype, opts=enable_speed_mem_opts, use_templates=use_templates,
           use_rna_msa=use_rna_msa)
print('Seeds:', SEEDS)
print(CFG)

## 5b. (Optional) Enable PAE capture — run BEFORE cell 6 to save dense PAE

In [ ]:
#@title B2-pre. (Optional) Enable PAE capture for the batch { display-mode: "form" }
# Run this BEFORE B2 if you want dense PAE matrices. It installs a tiny interpreter-startup patch
# that, as each job/seed is written during B2, also saves the expected PAE (Angstrom) as .npz next
# to the CIFs - no re-running. Sample order is forced to ORIGINAL so PAE/CIF/summary indices align
# (B3/B4 rank samples themselves, so this changes nothing downstream).
import os, textwrap

_patch_dir = '/content/_pae_patch'
os.makedirs(_patch_dir, exist_ok=True)
open(os.path.join(_patch_dir, 'sitecustomize.py'), 'w').write(textwrap.dedent('''
import os
if os.environ.get("PROTENIX_DUMP_PAE") == "1":
    try:
        import glob, numpy as np, torch
        import runner.inference as RI

        _orig_init = RI.InferenceRunner.init_dumper
        def _init_dumper(self, need_atom_confidence=False, sorted_by_ranking_score=True):
            # keep original sample order so pae_sample_i lines up with cif/summary sample_i
            return _orig_init(self, need_atom_confidence=need_atom_confidence,
                              sorted_by_ranking_score=False)
        RI.InferenceRunner.init_dumper = _init_dumper

        _orig_dump = RI.DataDumper.dump
        def _dump(self, *a, **k):
            out = _orig_dump(self, *a, **k)
            try:
                pred = k.get("pred_dict"); pdb_id = k.get("pdb_id"); seed = k.get("seed")
                if pred is not None and pred.get("pae") is not None:
                    lg = pred["pae"]
                    lg = lg if torch.is_tensor(lg) else torch.as_tensor(lg)
                    pr = torch.softmax(lg.float(), dim=-1)         # [S, N, N, bins]
                    nb = pr.shape[-1]
                    centers = (torch.arange(nb, dtype=torch.float32) + 0.5) * (32.0 / nb)
                    pae = (pr.cpu() * centers).sum(-1).numpy().astype("float16")   # [S, N, N] Angstrom
                    base = getattr(self, "base_dir", None) or getattr(self, "dump_dir", ".")
                    # Find THIS seed's output dir from the summary JSON just written by _orig_dump.
                    # (CIF names don't carry the seed; the seed lives in the folder name, e.g. seed_1.)
                    want = {str(seed), f"seed_{seed}", f"seed-{seed}"}
                    sjs = glob.glob(os.path.join(base, glob.escape(str(pdb_id)), "**",
                                    "*summary_confidence*.json"), recursive=True)
                    hit = [s for s in sjs if want & set(os.path.relpath(s, base).split(os.sep))] or sjs
                    if hit:
                        hit.sort(key=os.path.getmtime, reverse=True)
                        d = os.path.dirname(hit[0])
                        seed_dir = os.path.dirname(d) if os.path.basename(d) == "predictions" else d
                    else:
                        seed_dir = os.path.join(base, str(pdb_id), f"seed_{seed}")
                    pae_dir = os.path.join(seed_dir, "pae")     # -> <job>/seed_<seed>/pae
                    os.makedirs(pae_dir, exist_ok=True)
                    for i in range(pae.shape[0]):
                        np.savez_compressed(os.path.join(pae_dir,
                            f"{pdb_id}_{seed}_pae_sample_{i}.npz"), pae=pae[i])
                    print(f"[pae_dump] {pdb_id} seed {seed}: saved {pae.shape[0]} PAE matrices -> {pae_dir}")
            except Exception as e:
                print("[pae_dump] warning:", repr(e))
            return out
        RI.DataDumper.dump = _dump
        print("[pae_dump] active: PAE npz per sample (original order).")
    except Exception as e:
        print("[pae_dump] install failed:", repr(e))
'''))

os.environ['PROTENIX_DUMP_PAE'] = '1'
os.environ['PYTHONPATH'] = _patch_dir + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PAE capture ENABLED. Run B2 as usual; <job>_<seed>_pae_sample_N.npz appears next to each CIF.')
print('Disable with:  os.environ["PROTENIX_DUMP_PAE"] = "0"')


## 6. MSA + inference

If `use_msa` is on, this first calls `protenix msa` against the ColabFold server to build
paired/unpaired alignments and write them into a copy of the JSON, then runs `protenix predict`
on the MSA-annotated JSON. The MSA step adds a few minutes per unique sequence.

In [ ]:
#@title 6. MSA + inference { display-mode: "form" }
import os, glob, json, subprocess, shutil

assert JOB_SPEC is not None, 'Run one input cell (4a-4d) first.'
job = JOB_SPEC['name']

FORCE_REMSA = False   # set True to recompute MSAs even if a cached -update-msa.json exists

job_in  = os.path.join(INPUT_DIR, job + '.json')
job_msa = os.path.join(MSA_DIR, job)
import datetime
_stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RUN_DIR = os.path.join(OUT_DIR, f'run_{_stamp}_{job}')   # dated parent + job-name suffix
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(job_msa, exist_ok=True)
write_json(JOB_SPEC, job_in)
print('This run ->', RUN_DIR)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation OOM
VERBOSE = False   # set True to see all of Protenix's INFO logs

def _keep(line):
    if VERBOSE:
        return True
    low = line.lower()
    if any(k in low for k in ('error', 'traceback', 'failed', 'exception',
                              'out of memory', 'warning', 'file "', '^^^')):
        return True
    if any(k in line for k in ('completed', 'Job completed', 'N_token', 'Downloading',
                               'Saved', 'Seed', 'Predict', 'Inference')):
        return True
    if '%|' in line or 'it/s' in line or 's/it' in line:
        return True
    if 'INFO' in line:
        return False
    return True

def run(cmd, env=None):
    print('\n+ ' + ' '.join(cmd), flush=True)
    e = dict(os.environ)
    if env: e.update(env)
    p = subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    buf = []
    for line in p.stdout:
        buf.append(line)            # keep everything for retry-detection
        if _keep(line):
            print(line, end='')     # but only print the signal
    p.wait()
    return p.returncode, ''.join(buf)

def find_annotated(jin):
    # ONLY this job's MSA-annotated JSON. The job name is baked into the stem and into
    # job_msa/, so we never glob the shared INPUT_DIR (that matched OTHER jobs' caches
    # and could predict the previous job's sequences under the new job's name).
    pool = [os.path.splitext(jin)[0] + '-update-msa.json']
    pool += glob.glob(os.path.join(job_msa, '**', '*.json'), recursive=True)
    for c in pool:
        if os.path.exists(c):
            try:
                if 'MsaPath' in open(c).read():
                    return c
            except OSError:
                pass
    return None

predict_json = job_in

# ---- 1) MSA via ColabFold server (cached if already computed) ----
if CFG['use_msa']:
    cached = find_annotated(job_in)
    if cached and not FORCE_REMSA:
        predict_json = cached
        print('Reusing cached MSA JSON:', predict_json)
    else:
        rc, _ = run(['protenix', 'msa', '--input', job_in, '--out_dir', job_msa,
                     '--msa_server_mode', CFG['msa_server_mode']],
                    env={'MMSEQS_SERVICE_HOST_URL': 'https://api.colabfold.com'})
        if rc != 0:
            raise SystemExit('MSA step failed.')
        ann = find_annotated(job_in)
        if ann:
            predict_json = ann; print('\nUsing MSA-annotated JSON:', predict_json)
        else:
            print('\nWARNING: no MSA-annotated JSON found; predicting without MSA paths.')

# Templates need a local pdb_seqres DB + hmmer, which the online MSA route does not provide.
use_template = CFG['use_templates']
if use_template and 'templatesPath' not in open(predict_json).read():
    print('NOTE: no templatesPath in JSON / no local template DB -> disabling templates for this run.')
    use_template = False

# ---- 2) Inference (this build exposes the `pred` subcommand) ----
base = ['protenix', 'pred',
        '-i', predict_json, '-o', RUN_DIR,
        '--seeds', CFG['seeds_csv'],
        '-n', CFG['model_name'],
        '--use_msa', str(CFG['use_msa']).lower(),
        '--use_template', str(use_template).lower(),
        '--use_rna_msa', str(CFG['use_rna_msa']).lower(),
        '--dtype', CFG['dtype']]
extras = ['--sample_diffusion.N_sample', str(CFG['n_sample'])]
if CFG['use_tfg']:
    extras += ['--use_tfg_guidance', 'true']
if CFG['opts']:
    # Protenix ships its fused CUDA kernels precompiled only up to sm_100, so on a Blackwell
    # card (sm_120) --enable_fusion throws "no kernel image" - and because CUDA errors are
    # async it surfaces at the NEXT op (e.g. F.linear), masquerading as a core-arch problem.
    # Native ops do have sm_120, so we drop fusion there and keep the rest.
    if os.environ.get('TORCH_CUDA_ARCH_LIST', '') == '12.0':
        print('Blackwell (sm_120): disabling --enable_fusion (fused kernels are not built for '
              'sm_120; using native ops). --enable_cache kept.')
        extras += ['--enable_cache', 'true']
    else:
        extras += ['--enable_fusion', 'true', '--enable_cache', 'true']

rc, out = run(base + extras)
if rc != 0 and ('o such option' in out or 'unrecognized arguments' in out):
    print('\nRetrying without optional flags this build does not accept...')
    rc, out = run(base)
if rc != 0:
    raise SystemExit(f'Inference failed (exit {rc}).')
print('\nDone. Outputs under:', RUN_DIR)


## 7. Rank predictions by confidence

In [ ]:
#@title 7. Rank predictions by confidence { display-mode: "form" }
import glob, os, json
import pandas as pd, numpy as np

assert JOB_SPEC is not None, (
    "JOB_SPEC not set. For a BATCH: run B3 (it auto-selects the top hit) or set "
    "JOB_SPEC = {'name': '<job>'}. For a SINGLE run: run an input cell 4a-4d.")
_RUN = globals().get('RUN_DIR', OUT_DIR)
job_out = os.path.join(_RUN, JOB_SPEC['name'])
rows = []
for jf in glob.glob(os.path.join(job_out, '**', '*summary_confidence*.json'), recursive=True):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    # interface confidence = weakest off-diagonal chain-pair ipTM (None for monomers)
    cpi = np.array(d.get('chain_pair_iptm', []), dtype=float)
    if cpi.ndim == 2 and cpi.shape[0] > 1:
        iface = float(cpi[~np.eye(cpi.shape[0], dtype=bool)].min())
    else:
        iface = None
    seed = jf.split(os.sep)[-3] if os.sep + 'seed' in jf else ''
    rows.append({
        'seed': seed,
        'cif': cif if os.path.exists(cif) else '(missing)',
        'summary': jf,
        'ranking_score': d.get('ranking_score'),
        'iface_iptm': iface,
        'iptm': d.get('iptm'),
        'ptm': d.get('ptm'),
        'plddt': d.get('plddt'),
        'gpde': d.get('gpde'),
        'disorder': d.get('disorder'),
        'has_clash': d.get('has_clash'),
    })

assert rows, 'No summary_confidence JSONs found. Did cell 6 finish?'
df = pd.DataFrame(rows).sort_values('ranking_score', ascending=False, na_position='last').reset_index(drop=True)
pd.set_option('display.max_colwidth', 80)
display(df.drop(columns=['summary']))
BEST_CIF = df.iloc[0]['cif']
BEST_SUMMARY = df.iloc[0]['summary']
print('\nBest model:', BEST_CIF)
if df.iloc[0]['iface_iptm'] is not None:
    print(f"Interface ipTM of best model: {df.iloc[0]['iface_iptm']:.3f}  "
          "(>0.8 confident, 0.6-0.8 plausible, <0.6 weak)")

## 8. Model viewer (MolView) + scores

In [ ]:
#@title Model viewer (MolView) + scores { display-mode: "form" }
# Pick any predicted model from the dropdowns; it renders in MolView (Mol*-based) colored by
# pLDDT, with its confidence scores shown. Works for a single job or a whole batch.
import os, glob, json, re, subprocess, sys
import numpy as np, pandas as pd

try:
    import molview as mv
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'molview'], check=False)
    import molview as mv
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

_RUN = globals().get('RUN_DIR', OUT_DIR)

def _seed_of(p):
    for x in p.split(os.sep):
        if x.startswith('seed_'):
            return x.split('seed_')[-1]
    for x in p.split(os.sep):
        if x.isdigit():
            return x
    return 'NA'

def _interface_iptm(d):
    try:
        a = np.array(d.get('chain_pair_iptm'), dtype=float)
        if a.ndim == 2 and a.shape[0] > 1:
            off = a.copy(); np.fill_diagonal(off, np.nan)
            return float(np.nanmax(off))
    except Exception:
        pass
    return float('nan')

def _cif_for(jf):
    # same derivation cell 7 uses, with a glob fallback
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

# optional ipSAE table (present only after the ipSAE cell has run)
_ipsae = None
_ip_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
if os.path.exists(_ip_csv):
    try:
        _ipsae = pd.read_csv(_ip_csv)
        _ipsae['seed'] = _ipsae['seed'].astype(str)
        _ipsae['ipSAE'] = pd.to_numeric(_ipsae.get('ipSAE'), errors='coerce')
    except Exception:
        _ipsae = None

# discover models the same way cell 7 / B-cells do: via the summary JSONs
recs = []
for jf in glob.glob(os.path.join(_RUN, '**', '*summary_confidence*.json'), recursive=True):
    try:
        d = json.load(open(jf))
    except Exception:
        continue
    cif = _cif_for(jf)
    if not cif:
        continue
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    recs.append(dict(job=os.path.relpath(jf, _RUN).split(os.sep)[0],
                     seed=_seed_of(jf), sample=int(ms.group(1)) if ms else 0,
                     cif=cif, ranking=d.get('ranking_score', float('nan')),
                     iptm=d.get('iptm', float('nan')), ptm=d.get('ptm', float('nan')),
                     iiptm=_interface_iptm(d), plddt=d.get('plddt'),
                     has_clash=d.get('has_clash')))
mdf = pd.DataFrame(recs)

if mdf.empty:
    print('No models found under', _RUN)
    print('Looked for *summary_confidence*.json - has the inference cell finished and written here?')
else:
    mdf = mdf.sort_values('ranking', ascending=False).reset_index(drop=True)
    jobs = list(dict.fromkeys(mdf['job']))            # best-ranked job first
    job_dd   = widgets.Dropdown(options=jobs, description='Job:',
                                layout=widgets.Layout(width='70%'))
    model_dd = widgets.Dropdown(description='Model:', layout=widgets.Layout(width='70%'))
    color_dd = widgets.Dropdown(options=['plddt', 'chain', 'rainbow', 'secondary'],
                                value='plddt', description='Color:',
                                layout=widgets.Layout(width='40%'))
    out = widgets.Output()

    def _opts(job):
        sub = mdf[mdf['job'] == job]
        return [(f"seed {r['seed']} \u00b7 sample {r['sample']}  |  ipTM {r['iptm']:.2f}  "
                 f"pTM {r['ptm']:.2f}  rank {r['ranking']:.3f}", r['cif'])
                for _, r in sub.iterrows()]

    def _render(cif):
        with out:
            clear_output(wait=True)
            r = mdf[mdf['cif'] == cif].iloc[0]
            row = {'plddt': r['plddt'], 'pTM': r['ptm'], 'ipTM': r['iptm'],
                   'interface_ipTM': r['iiptm'], 'ranking_score': r['ranking'],
                   'has_clash': r['has_clash']}
            if _ipsae is not None:
                sel = _ipsae[(_ipsae['job'] == r['job']) & (_ipsae['seed'] == str(r['seed']))]
                if len(sel) and 'ipSAE' in sel.columns:
                    row['ipSAE (max pair)'] = float(sel['ipSAE'].max())
            print(f"{r['job']}  |  seed {r['seed']}  \u00b7  sample {r['sample']}")
            display(pd.DataFrame([row]).T.rename(columns={0: 'value'}))
            try:
                v = mv.view(width=720, height=520, panel=True)
                with open(cif) as f:
                    v.addModel(f.read())
                v.setColorMode(color_dd.value)
                v.show()
            except Exception as e:
                print('MolView render failed:', e)
                print('Open this file in ChimeraX instead:', cif)

    def _on_job(ch):
        if ch['name'] == 'value':
            model_dd.options = _opts(ch['new'])
            if model_dd.options:
                model_dd.value = model_dd.options[0][1]

    def _on_model(ch):
        if ch['name'] == 'value' and ch['new']:
            _render(ch['new'])

    def _on_color(ch):
        if ch['name'] == 'value' and model_dd.value:
            _render(model_dd.value)

    job_dd.observe(_on_job, names='value')
    model_dd.observe(_on_model, names='value')
    color_dd.observe(_on_color, names='value')

    display(widgets.HBox([job_dd]), widgets.HBox([model_dd, color_dd]), out)
    model_dd.options = _opts(jobs[0])
    if model_dd.options:
        model_dd.value = model_dd.options[0][1]
        _render(model_dd.value)


## 9. Per-residue pLDDT + interface confidence

Protenix's CLI does not export a dense PAE matrix — its confidence output is the summary JSON
(scalars + chain-level matrices). So the left panel is per-residue pLDDT (averaged from the CIF
B-factors via `gemmi`, chain boundaries marked) and the right panel is `chain_pair_iptm`, the
pairwise inter-chain ipTM that tells you how confident the model is in each interface.

In [ ]:
#@title 9. Per-residue pLDDT + interface confidence { display-mode: "form" }
import gemmi, numpy as np, json, os
import matplotlib.pyplot as plt

assert 'BEST_CIF' in globals() and 'BEST_SUMMARY' in globals(), \
    'Run cell 7 first - it sets BEST_CIF / BEST_SUMMARY (after a batch, run B3 then 7).'

# ---- left: per-residue pLDDT from CIF B-factors ----
st = gemmi.read_structure(BEST_CIF)
model = st[0]
res_plddt, boundaries, chain_mid, chain_names, idx = [], [], [], [], 0
for chain in model:
    start = idx
    for res in chain:
        bs = [a.b_iso for a in res]
        if bs:
            res_plddt.append(sum(bs)/len(bs)); idx += 1
    if idx > start:
        boundaries.append(idx); chain_mid.append((start+idx)//2); chain_names.append(chain.name)
res_plddt = np.array(res_plddt)

# ---- right: interface confidence (chain_pair_iptm) from the summary JSON ----
d = json.load(open(BEST_SUMMARY))
cpi = np.array(d.get('chain_pair_iptm', []), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(range(1, len(res_plddt)+1), res_plddt, lw=1.1, color='#2b6cb0')
ax.axhspan(90,100,alpha=.07,color='blue');  ax.axhspan(70,90,alpha=.07,color='cyan')
ax.axhspan(50,70,alpha=.07,color='orange'); ax.axhspan(0,50,alpha=.07,color='red')
for b in boundaries[:-1]:
    ax.axvline(b+0.5, color='k', ls='--', lw=.6, alpha=.5)
for m, nm in zip(chain_mid, chain_names):
    ax.text(m, 103, nm, ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Residue'); ax.set_ylabel('pLDDT'); ax.set_ylim(0, 108)
ax.set_title(f'Per-residue pLDDT (mean {res_plddt.mean():.1f})')

ax2 = axes[1]
if cpi.ndim == 2 and cpi.size:
    im = ax2.imshow(cpi, cmap='viridis', vmin=0, vmax=1)
    for i in range(cpi.shape[0]):
        for j in range(cpi.shape[1]):
            ax2.text(j, i, f'{cpi[i,j]:.2f}', ha='center', va='center', fontsize=10,
                     color='white' if cpi[i,j] < 0.6 else 'black')
    labs = chain_names if len(chain_names) == cpi.shape[0] else [str(i) for i in range(cpi.shape[0])]
    ax2.set_xticks(range(len(labs))); ax2.set_xticklabels(labs)
    ax2.set_yticks(range(len(labs))); ax2.set_yticklabels(labs)
    fig.colorbar(im, ax=ax2, label='chain-pair ipTM')
    ax2.set_title('Interface confidence (chain_pair_iptm)')
else:
    ax2.axis('off'); ax2.set_title('chain_pair_iptm not available (monomer?)')
plt.tight_layout(); plt.show()

if cpi.ndim == 2 and cpi.shape[0] > 1:
    off = cpi[~np.eye(cpi.shape[0], dtype=bool)]
    print(f'Inter-chain ipTM: min {off.min():.3f}, max {off.max():.3f}  '
          '(>0.8 confident interface, 0.6-0.8 plausible, <0.6 weak/unreliable)')
    print('Tip: agreement of the top interface across seeds matters more than any single value.')

## 9b. Interface contacts & residue-residue reliability (PAE substitute)

Protenix doesn't export PAE, so this is the practical replacement for judging *specific*
residue-residue interactions. It builds the inter-chain contact map for the top model, lists the
closest residue pairs with their pLDDT, and — the key part — reports how often each contact recurs
across all samples/seeds. A contact that appears in nearly every model at high pLDDT is a
trustworthy predicted interaction; one that flickers in and out is not. Exports one contacts
table per job (`<job>/interface_contacts.csv`) plus a combined `interface_contacts_all_jobs.csv`.

In [ ]:
#@title 9b. Interface contacts & residue-residue reliability (+ per-job table) { display-mode: "form" }
# Builds the inter-chain contact map for each job's best model, lists the closest residue pairs
# with their pLDDT, and reports how often each contact recurs across that job's samples/seeds.
# Exports one table per job (and a combined CSV); shows the map + table for the current top job.
import gemmi, numpy as np, glob, os, json, re
import pandas as pd
import matplotlib.pyplot as plt

_RUN = globals().get('RUN_DIR', OUT_DIR)
CONTACT = 8.0          #@param {type:"number"}
max_rows_print = 30    #@param {type:"integer"}

def load_chains(cif):
    st = gemmi.read_structure(cif); m = st[0]
    out = {}
    for ch in m:
        rl = []
        for res in ch:
            coords = np.array([[a.pos.x, a.pos.y, a.pos.z] for a in res], dtype=float)
            if len(coords) == 0:
                continue
            ca = [[a.pos.x, a.pos.y, a.pos.z] for a in res if a.name == 'CA']
            rep = np.array(ca[0]) if ca else coords.mean(0)
            tab = gemmi.find_tabulated_residue(res.name)
            code = tab.one_letter_code.upper() if tab else 'X'
            rl.append({'num': res.seqid.num, 'code': code, 'rep': rep,
                       'plddt': float(np.mean([a.b_iso for a in res]))})
        if rl:
            out[ch.name] = rl
    return out

def _best_cif(job):
    best, br = None, -1e9
    for jf in glob.glob(os.path.join(_RUN, job, '**', '*summary_confidence*sample_*.json'), recursive=True):
        try:
            r = float(json.load(open(jf)).get('ranking_score') or 0)
        except Exception:
            r = 0.0
        if r > br:
            cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
            if not os.path.exists(cif):
                ms = re.search(r'sample_(\d+)', os.path.basename(jf)); d = os.path.dirname(jf)
                c = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
                     else glob.glob(os.path.join(d, '*.cif')))
                cif = c[0] if c else None
            if cif:
                br, best = r, cif
    return best

def contacts_table(job, cutoff=CONTACT):
    """Unique inter-chain residue pairs < cutoff for the job's best model, with recurrence
       across the job's samples. Returns (rows, D, A, B, cA, cB, n_used)."""
    cif = _best_cif(job)
    if not cif:
        return [], None, None, None, None, None, 0
    chains = load_chains(cif); names = list(chains)
    if len(names) < 2:
        return [], None, None, None, None, None, 0
    cA, cB = names[0], names[1]                     # first two chains; edit for another pair
    A, B = chains[cA], chains[cB]
    PA = np.array([r['rep'] for r in A]); PB = np.array([r['rep'] for r in B])
    D = np.linalg.norm(PA[:, None, :] - PB[None, :, :], axis=-1)
    freq = np.zeros_like(D); n_used = 0
    for cf in glob.glob(os.path.join(_RUN, job, '**', '*sample_*.cif'), recursive=True):
        ch = load_chains(cf)
        if cA in ch and cB in ch and len(ch[cA]) == len(A) and len(ch[cB]) == len(B):
            pa = np.array([r['rep'] for r in ch[cA]]); pb = np.array([r['rep'] for r in ch[cB]])
            freq += (np.linalg.norm(pa[:, None, :] - pb[None, :, :], axis=-1) < cutoff); n_used += 1
    freq /= max(n_used, 1)
    ii, jj = np.where(D < cutoff)
    rows, seen = [], set()
    for i, j in sorted(zip(ii, jj), key=lambda p: D[p]):
        key = (A[i]['num'], B[j]['num'])
        if key in seen:
            continue
        seen.add(key)
        rows.append(dict(job=job, chainA=cA, resA=A[i]['num'], codeA=A[i]['code'],
                         plddtA=round(A[i]['plddt'], 1), chainB=cB, resB=B[j]['num'],
                         codeB=B[j]['code'], plddtB=round(B[j]['plddt'], 1),
                         distance=round(float(D[i, j]), 2), recurrence=round(float(freq[i, j]), 3)))
    return rows, D, A, B, cA, cB, n_used

# ---- per-job export across every job in the run ----
jobs = sorted(d for d in os.listdir(_RUN)
              if os.path.isdir(os.path.join(_RUN, d)) and not d.startswith('_'))
all_rows = []
for job in jobs:
    rows = contacts_table(job)[0]
    if rows:
        all_rows += rows
        pd.DataFrame(rows).to_csv(os.path.join(_RUN, job, 'interface_contacts.csv'), index=False)

if all_rows:
    df = pd.DataFrame(all_rows)
    out_csv = os.path.join(_RUN, 'interface_contacts_all_jobs.csv')
    df.to_csv(out_csv, index=False)
    print(f'Wrote {out_csv}  |  {len(df)} contacts across {df["job"].nunique()} job(s).')
    print('Per-job copies: <job>/interface_contacts.csv')
else:
    df = None
    print('No inter-chain contacts found (monomers, or no >=2-chain jobs).')

# ---- interactive map + table for the current top job ----
_js = globals().get('JOB_SPEC') or {}
view_job = (_js.get('name') if _js.get('name') in jobs
            else (df.iloc[0]['job'] if df is not None else (jobs[0] if jobs else None)))
if view_job:
    rows, D, A, B, cA, cB, n_used = contacts_table(view_job)
    if D is not None:
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(D, cmap='viridis_r', vmin=0, vmax=20, aspect='auto')
        ax.set_xlabel(f'chain {cB} residue'); ax.set_ylabel(f'chain {cA} residue')
        ax.set_title(f'{view_job}: inter-chain distances | contacts < {CONTACT} A '
                     f'(recurrence over {n_used} models)')
        fig.colorbar(im, ax=ax, label='distance (Angstrom)'); plt.tight_layout(); plt.show()
        from IPython.display import display
        print(f'\nTop {min(max_rows_print, len(rows))} of {len(rows)} contacts for {view_job} '
              f'(chains {cA}-{cB}):\n')
        display(pd.DataFrame(rows).head(max_rows_print))
        print('\nHigh recurrence (~1.0) + high pLDDT on both partners = trustworthy specific contact.')


## 9c. PAE export — ChimeraX npz + heatmap (best per seed)

In [ ]:
#@title B5. PAE - best per seed: ChimeraX/ipSAE npz + heatmap { display-mode: "form" }
# For each (job, seed) picks the top-ranked sample and writes, into that seed's pae/ folder:
#   pae_<cifstem>.npz        (key 'pae', per-token; ligand-safe input for ChimeraX>=1.10 + ipSAE)
#   <job>_<seed>_pae_best.npz, _pae.json (ColabFold schema), _pae.png (heatmap w/ chain lines)
# Models are located via the summary JSONs (CIF derived from them), so naming/layout don't matter.
import os, glob, json, re
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
try:
    import gemmi; _HAS_GEMMI = True
except Exception:
    _HAS_GEMMI = False

_RUN = globals().get('RUN_DIR', OUT_DIR)
prune_others = True   #@param {type:"boolean"}
round_dp = 2          #@param {type:"integer"}

def _seed_of(p):
    for x in p.split(os.sep):
        if x.startswith('seed_'):
            return x.split('seed_')[-1]
    for x in p.split(os.sep):
        if x.isdigit():
            return x
    return 'NA'

def _cif_for(jf):
    cif = jf.replace('summary_confidence_', '').replace('.json', '.cif')
    if os.path.exists(cif):
        return cif
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    d = os.path.dirname(jf)
    cand = (glob.glob(os.path.join(d, f'*sample_{ms.group(1)}.cif')) if ms
            else glob.glob(os.path.join(d, '*.cif')))
    return cand[0] if cand else None

def _seed_dir_of(cif):
    d = os.path.dirname(cif)
    return os.path.dirname(d) if os.path.basename(d) == 'predictions' else d

def _chain_boundaries(cif):
    if not (_HAS_GEMMI and cif and os.path.exists(cif)):
        return []
    try:
        st = gemmi.read_structure(cif); m = st[0]
        bnd, c = [], 0
        for ch in m:
            c += sum(1 for _ in ch); bnd.append(c)
        return bnd[:-1]
    except Exception:
        return []

def _heatmap(arr, cif, title, path):
    bnd = _chain_boundaries(cif)
    plt.figure(figsize=(5.2, 4.4))
    im = plt.imshow(arr, cmap='Greens_r', vmin=0, vmax=32)
    for b in bnd:
        plt.axhline(b - 0.5, color='k', lw=0.6); plt.axvline(b - 0.5, color='k', lw=0.6)
    plt.colorbar(im, label='PAE (Angstrom)')
    plt.title(title); plt.xlabel('Scored token'); plt.ylabel('Aligned token')
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.close()

# best sample per (job, seed) via the summary JSONs
best = {}
for jf in glob.glob(os.path.join(_RUN, '**', '*summary_confidence*sample_*.json'), recursive=True):
    job = os.path.relpath(jf, _RUN).split(os.sep)[0]
    seed = _seed_of(jf)
    ms = re.search(r'sample_(\d+)', os.path.basename(jf))
    if not ms:
        continue
    try:
        rank = float(json.load(open(jf)).get('ranking_score') or 0)
    except Exception:
        rank = 0.0
    k = (job, seed)
    if k not in best or rank > best[k]['rank']:
        best[k] = {'rank': rank, 'jf': jf, 'N': ms.group(1)}

consolidated = os.path.join(_RUN, '_pae'); os.makedirs(consolidated, exist_ok=True)
n_npz = n_json = n_png = 0
for (job, seed), r in sorted(best.items()):
    cif = _cif_for(r['jf'])
    if not cif:
        continue
    seed_dir = _seed_dir_of(cif)
    pae_dir = os.path.join(seed_dir, 'pae')
    src = os.path.join(pae_dir, f'{job}_{seed}_pae_sample_{r["N"]}.npz')
    if not os.path.exists(src):
        cand = glob.glob(os.path.join(pae_dir, f'*_pae_sample_{r["N"]}.npz'))
        src = cand[0] if cand else None
    if not src or not os.path.exists(src):
        continue
    os.makedirs(pae_dir, exist_ok=True)
    arr = np.load(src)['pae'].astype('float32')
    cifstem = os.path.basename(cif)[:-4]

    np.savez_compressed(os.path.join(pae_dir, f'pae_{cifstem}.npz'), pae=arr.astype('float32'))   # ChimeraX/ipSAE
    np.savez_compressed(os.path.join(pae_dir, f'{job}_{seed}_pae_best.npz'), pae=arr.astype('float16'))
    json.dump([{"predicted_aligned_error": np.round(arr, round_dp).tolist(),
                "max_predicted_aligned_error": float(arr.max())}],
              open(os.path.join(pae_dir, f'{job}_{seed}_pae.json'), 'w'))
    _heatmap(arr, cif, f'{job}  seed {seed}', os.path.join(pae_dir, f'{job}_{seed}_pae.png'))
    np.savez_compressed(os.path.join(consolidated, f'{job}_seed{seed}_pae.npz'), pae=arr.astype('float16'))
    n_npz += 1; n_json += 1; n_png += 1

print(f'Best-per-seed: {n_npz} Boltz/ChimeraX npz + {n_json} JSON + {n_png} heatmaps, each in <job>/seed_<seed>/pae/.')
print('Consolidated npz copies ->', consolidated)

if prune_others:
    removed = 0
    for p in glob.glob(os.path.join(_RUN, '**', '*_pae_sample_*.npz'), recursive=True):
        try:
            os.remove(p); removed += 1
        except Exception:
            pass
    print(f'Pruned {removed} per-sample PAE files (kept best per seed).')

if best:
    (job, seed), r = max(best.items(), key=lambda kv: kv[1]['rank'])
    png = glob.glob(os.path.join(_RUN, job, '**', f'{job}_{seed}_pae.png'), recursive=True)
    if png:
        from IPython.display import Image, display
        print(f'Top model: {job} seed {seed} (rank {r["rank"]:.3f})')
        display(Image(filename=png[0]))
print('ChimeraX >=1.10 (ligand-safe):  open <cif>  then  open pae_<cifstem>.npz structure #1')
print('ChimeraX older / protein-only:  open <cif>  then  open <job>_<seed>_pae.json format pae')


## 9d. ipSAE interface scores (Dunbrack, Boltz mode)

In [ ]:
#@title B6. ipSAE interface scores (Dunbrack, Boltz mode) { display-mode: "form" }
# Runs Dunbrack's ipsae.py on each best-per-seed model using the Boltz-style npz + cif
# (per-token, ligand-aware). Collects chain-pair ipSAE / ipTM / pDockQ / LIS into a CSV.
import os, glob, json, shutil, subprocess, sys
import numpy as np, pandas as pd

pae_cutoff  = 10   #@param {type:"number"}
dist_cutoff = 10   #@param {type:"number"}
_RUN = globals().get('RUN_DIR', OUT_DIR)

# --- fetch ipsae.py once (Colab can reach raw.githubusercontent.com) ---
IPSAE = '/content/ipsae.py'
if not (os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000):
    for url in ('https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py',
                'https://raw.githubusercontent.com/DunbrackLab/IPSAE/master/ipsae.py'):
        os.system(f'wget -q -O {IPSAE} "{url}"')
        if os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000:
            break
assert os.path.exists(IPSAE) and os.path.getsize(IPSAE) > 5000, 'Could not download ipsae.py'
print('ipsae.py ready:', os.path.getsize(IPSAE), 'bytes')

def _seed_of(path):
    for p in path.split(os.sep):
        if p.startswith('seed_'):
            return p.split('seed_')[-1]
    for p in path.split(os.sep):
        if p.isdigit():
            return p
    return 'NA'

ps = f'{int(pae_cutoff):02d}'
ds = f'{int(dist_cutoff):02d}'
rows, n_run, n_fail = [], 0, 0

for npz in sorted(glob.glob(os.path.join(_RUN, '**', 'pae', 'pae_*.npz'), recursive=True)):
    if os.path.basename(npz).endswith('_pae_best.npz'):
        continue
    cifstem = os.path.basename(npz)[4:-4]                 # strip 'pae_' and '.npz'
    job = os.path.relpath(npz, _RUN).split(os.sep)[0]
    seed = _seed_of(npz)
    seed_dir = os.path.dirname(os.path.dirname(npz))      # .../seed_<seed>  (parent of pae/)
    cifs = glob.glob(os.path.join(seed_dir, '**', cifstem + '.cif'), recursive=True)
    if not cifs:
        continue
    cif = cifs[0]; cifdir = os.path.dirname(cif); pae_dir = os.path.dirname(npz)

    r = subprocess.run([sys.executable, IPSAE, npz, cif, str(pae_cutoff), str(dist_cutoff)],
                       capture_output=True, text=True)
    n_run += 1
    out_txt = os.path.join(cifdir, f'{cifstem}_{ps}_{ds}.txt')
    if not os.path.exists(out_txt):
        alt = glob.glob(os.path.join(cifdir, f'{cifstem}_*_*.txt'))
        alt = [a for a in alt if not a.endswith('_byres.txt')]
        out_txt = alt[0] if alt else None
    if not out_txt or not os.path.exists(out_txt):
        n_fail += 1
        if n_fail <= 3:
            print(f'  no ipSAE output: {cifstem}\n   {r.stderr.strip()[-300:]}')
        continue

    # tidy: move ipsae outputs (.txt/_byres.txt/.pml) into the pae/ folder
    for f in glob.glob(os.path.join(cifdir, f'{cifstem}_{ps}_{ds}*')):
        try:
            shutil.move(f, os.path.join(pae_dir, os.path.basename(f)))
        except Exception:
            pass
    txt = os.path.join(pae_dir, os.path.basename(out_txt))
    if not os.path.exists(txt):
        txt = out_txt

    with open(txt) as fh:
        lines = [l.rstrip() for l in fh if l.strip()]
    hdr = next((i for i, l in enumerate(lines) if 'ipSAE' in l and ('Chn1' in l or l.split()[0] == 'Chn1')), None)
    if hdr is None:
        continue
    cols = lines[hdr].split()
    for l in lines[hdr + 1:]:
        parts = l.split()
        if len(parts) != len(cols):
            continue
        d = dict(zip(cols, parts)); d['job'] = job; d['seed'] = seed
        rows.append(d)

print(f'ipSAE run on {n_run} models ({n_fail} without output).')

if rows:
    df = pd.DataFrame(rows)
    keep_str = {'job', 'seed', 'Chn1', 'Chn2', 'Type', 'Model'}
    for c in df.columns:
        if c not in keep_str:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().any():
                df[c] = conv
    lead = [c for c in ['job', 'seed', 'Chn1', 'Chn2'] if c in df.columns]
    df = df[lead + [c for c in df.columns if c not in lead]]
    out_csv = os.path.join(_RUN, 'ipsae_all_chain_pairs.csv')
    df.to_csv(out_csv, index=False)
    print('Wrote', out_csv, '| rows:', len(df))

    if 'ipSAE' in df.columns:
        best = (df.sort_values('ipSAE', ascending=False)
                  .groupby('job', as_index=False).head(1)
                  .sort_values('ipSAE', ascending=False))
        best.to_csv(os.path.join(_RUN, 'ipsae_best_per_job.csv'), index=False)
        show = [c for c in ['job', 'seed', 'Chn1', 'Chn2', 'ipSAE', 'ipSAE_d0chn',
                            'ipTM_d0chn', 'pDockQ', 'LIS'] if c in best.columns]
        from IPython.display import display
        display(best[show].head(40))
else:
    print('No ipSAE rows parsed - check a sample .txt in a pae/ folder and the stderr above.')


## 10. Zip & download all outputs

In [ ]:
#@title 10. Zip & download all outputs { display-mode: "form" }
import shutil, os
job = JOB_SPEC['name']
zip_base = os.path.join(WORK, job + '_protenix_v2')
_RUN = globals().get('RUN_DIR', OUT_DIR)
shutil.make_archive(zip_base, 'zip', os.path.join(_RUN, job))
zp = zip_base + '.zip'
print('Zipped:', zp, f'({os.path.getsize(zp)/1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print('Download manually from the Files panel:', zp, '|', e)

## 10b. Archive this run (dated, seed-labelled - prevents rerun overwrites)

In [ ]:
#@title 10b. Archive this run (date prefix + seed suffix, no overwrites) { display-mode: "form" }
# Protenix overwrites OUT_DIR/<job>/seed_*/predictions/<job>_sample_N.* on every rerun, so reruns
# clobber earlier results. This copies the current outputs into a timestamped archive folder with
# self-describing names: <STAMP>_<job>_sample<N>_seed<seed>.<ext>  (date prefix, seed suffix).
# Originals are left untouched. Works after a single run (JOB_SPEC) or a batch (BATCH_NAMES).
import os, glob, shutil, datetime, zipfile

make_zip = True  #@param {type:"boolean"}

STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M')

if 'BATCH_NAMES' in globals() and BATCH_NAMES:
    jobs = list(BATCH_NAMES)
elif 'JOB_SPEC' in globals() and JOB_SPEC:
    jobs = [JOB_SPEC['name']]
else:
    raise SystemExit('Nothing to archive: run a batch (B1/B2) or set JOB_SPEC first.')

_RUN = globals().get('RUN_DIR', OUT_DIR)
archive_root = os.path.join(OUT_DIR, '_archive', STAMP)
n = 0
for nm in jobs:
    jdir = os.path.join(_RUN, nm)
    if not os.path.isdir(jdir):
        continue
    for f in glob.glob(os.path.join(jdir, '**', '*'), recursive=True):
        if not os.path.isfile(f):
            continue
        seed = next((p.split('seed_')[-1] for p in f.split(os.sep) if p.startswith('seed_')), 'NA')
        stem, ext = os.path.splitext(os.path.basename(f))
        dst_dir = os.path.join(archive_root, nm)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(f, os.path.join(dst_dir, f'{STAMP}_{stem}_seed{seed}{ext}'))
        n += 1

print(f'Archived {n} files from {len(jobs)} job(s) -> {archive_root}')

if make_zip:
    zpath = archive_root + '.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(archive_root, '**', '*'), recursive=True):
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, archive_root))
    print(f'Zipped -> {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB)')
    try:
        from google.colab import files
        files.download(zpath)
    except Exception as e:
        print('Download from the Files panel:', zpath, '|', e)
